In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

# Paths
vm_path = '/content/document-classifier'
model_path = os.path.join(vm_path, 'src/models/trained_models/best_resnet18.pth')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")
print(f"Model path: {model_path}")

Using device: cuda
Model path: /content/document-classifier/src/models/trained_models/best_resnet18.pth


In [2]:
# Load weights (this is already the state_dict)
state_dict = torch.load(model_path, map_location=device)

# Recreate model architecture
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)

# Load weights
model.load_state_dict(state_dict)

# Move to device
model = model.to(device)

# Eval mode
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [3]:
# Convert to feature extractor
model.fc = nn.Identity()

print("Converted to feature extractor")


# Test with one image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Replace this with an actual image path from your dataset
sample_image_path = "/content/drive/MyDrive/ColabNotebooks/document-classifier/data/test/myresume.png"

image = Image.open(sample_image_path).convert("RGB")
image = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    features = model(image)

print("Feature shape:", features.shape)

Converted to feature extractor
Feature shape: torch.Size([1, 512])


In [4]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 59.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 44.7 MB/s eta 0:00:00


In [5]:
import easyocr
import cv2
import numpy as np
import re


reader = easyocr.Reader(['en'])

def preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.equalizeHist(gray)

    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 2
    )
    kernel = np.ones((2,2), np.uint8)
    thresh = cv2.dilate(thresh, kernel, iterations=1)

    return thresh



def split_long_word(word):
    # split if word is too long and messy
    if len(word) > 12:
        return re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', word)
    return [word]


def clean_text(words):
    cleaned = []

    for w in words:
        w = w.strip()

        # remove weird characters
        w = re.sub(r'[^a-zA-Z0-9]', '', w)

        # split camelCase / PascalCase
        w = re.sub(r'([a-z])([A-Z])', r'\1 \2', w)

        split_words = []
        for part in w.split():
            split_words.extend(split_long_word(part))

        for sw in split_words:
            if len(sw) < 3:
                continue

            alpha_ratio = sum(c.isalpha() for c in sw) / len(sw)
            if alpha_ratio < 0.6:
                continue

            cleaned.append(sw)

    return cleaned

def remove_duplicates(words):
    seen = set()
    result = []
    for w in words:
        if w not in seen:
            seen.add(w)
            result.append(w)
    return result

def extract_text(image_path):
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError("Image not found")

    proc = preprocess(img)

    result1 = reader.readtext(img)
    result2 = reader.readtext(proc)

    result = result1 + result2

    result = [item for item in result if item[2] > 0.25]
    result = sorted(result, key=lambda x: x[0][0][1])

    texts = [item[1] for item in result]

    texts = clean_text(texts)
    texts = remove_duplicates(texts)

    return " ".join(texts)


text = extract_text("/content/drive/MyDrive/ColabNotebooks/document-classifier/data/test/Appily High School Resume.png")

print("===== FINAL CLEANED OCR OUTPUT =====")
print(text)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete===== FINAL CLEANED OCR OUTPUT =====
RACHEL BEAN responsiblehighschoolstudentlookingtofurtherher professionalexperience Contact Education Chicago Public Schools rbeanexamplecom High School Diploma 39GPA Wrachelbeanportfoliocom Activities 123Street Creative Writing Club Skills Member President Leadweeklymeetings Maintainreadinglist Alskillscertification Concact Keeptimeforeachreadingfairlydistributingtime Communication betweeneveryreaderforthemeeting Planclubeventsatleastonepersemester Leadershipandplanning The Literary Magazine Customerservice Reader Reviewallsubmissionsfortheprintingofthemagazinein Hobbies timelyfashion Collaboratewithfellowreaderstorateandagreeupon submission Work Experience Softball Hair Salon 2023present Additional Stylists Assistant Cheerfullygreetcustomersandbringthemtotheir stylistschair Babysitting Providecoffeeorwatertoguests Addlerfamilyhasthree Prepguestforhaircutswashconditionhead

In [6]:
import torch
from transformers import BertTokenizer, BertModel

# load pretrained BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

model.eval()  # inference mode


def extract_text_features(text):
    # tokenize input
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # use CLS token embedding (standard practice)
    cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: (1, 768)

    return cls_embedding


# 🔥 test with your OCR output
text_features = extract_text_features(text)

print("Text feature shape:", text_features.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Text feature shape: torch.Size([1, 768])


In [7]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

DATA_DIR = "/content/drive/MyDrive/ColabNotebooks/document-classifier/data/scanned_docs"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# full dataset (NO train/test folders exist)
dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

# split (same logic you already used before)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print("Classes:", dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Classes: ['ADVE', 'Email', 'Form', 'Letter', 'Memo', 'News', 'Note', 'Report', 'Resume', 'Scientific']
Train size: 2785
Val size: 697


In [ ]:
from tqdm import tqdm
import os
import torch
from torch.utils.data import Dataset

# ==================== CONFIG (UNCHANGED) ====================
CACHE_DIR = "/content/drive/MyDrive/document_classifier/cache"
TEXT_CACHE_DIR = os.path.join(CACHE_DIR, "text")
FEAT_CACHE_DIR = os.path.join(CACHE_DIR, "features")

os.makedirs(TEXT_CACHE_DIR, exist_ok=True)
os.makedirs(FEAT_CACHE_DIR, exist_ok=True)

# Global EasyOCR reader (kept for consistency)
reader = easyocr.Reader(['en'], gpu=True)

# ==================== HELPER FUNCTION (UNCHANGED) ====================
def check_empty_texts():
    empty_count = 0
    short_files = []
    for f in os.listdir(TEXT_CACHE_DIR):
        if f.endswith('.txt'):
            path = os.path.join(TEXT_CACHE_DIR, f)
            with open(path, 'r', encoding='utf-8') as file:
                content = file.read().strip()
                if len(content) < 5:
                    empty_count += 1
                    short_files.append(f)
    print(f"\n📋 Empty / Very Short Text Files: {empty_count}")
    if empty_count > 0:
        print("Some examples:", short_files[:5])
    return empty_count

# ==================== OPTIMIZED FUSION DATASET ====================
class FusionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_dataset = datasets.ImageFolder(root_dir, transform=transform)
        self.classes = self.image_dataset.classes
        self.transform = transform
        self.text_cache = {}
        
        print(f"🔍 Scanning cache for {len(self.image_dataset)} images...")

        # === FAST CACHE SCAN (This was the slow part before) ===
        try:
            cached_pt_files = {f for f in os.listdir(FEAT_CACHE_DIR) if f.endswith('.pt')}
        except:
            cached_pt_files = set()

        cached_count = 0
        for idx in range(len(self.image_dataset)):
            img_path, _ = self.image_dataset.samples[idx]
            base_name = os.path.basename(img_path)
            if base_name + ".pt" in cached_pt_files:
                cached_count += 1

        print(f"📊 Cache Status: {cached_count} / {len(self.image_dataset)} images already processed ({cached_count/len(self.image_dataset)*100:.1f}%)")
        print(f"⏭️ Remaining to process: {len(self.image_dataset) - cached_count} images\n")

        # Check empty texts
        check_empty_texts()

        print("🚀 Loading all cached features into memory (this may take 1-3 minutes)...\n")

        # === FAST LOADING OF CACHED FEATURES ===
        for idx in tqdm(range(len(self.image_dataset)), desc="Loading Features"):
            img_path, _ = self.image_dataset.samples[idx]
            base_name = os.path.basename(img_path)
            feat_file = os.path.join(FEAT_CACHE_DIR, base_name + ".pt")

            if os.path.exists(feat_file):
                try:
                    self.text_cache[img_path] = torch.load(feat_file, map_location='cpu')
                except Exception as e:
                    print(f"⚠️ Corrupted file {base_name}: {e} → Using zero vector")
                    self.text_cache[img_path] = torch.zeros(768)
            else:
                print(f"⚠️ Missing cache for {base_name} → Using zero vector")
                self.text_cache[img_path] = torch.zeros(768)

        print(f"\n✅ FusionDataset successfully loaded! All {len(self.text_cache)} features are ready.")

    def __len__(self):
        return len(self.image_dataset)

    def __getitem__(self, idx):
        image, label = self.image_dataset[idx]
        img_path, _ = self.image_dataset.samples[idx]
        text_feat = self.text_cache[img_path]
        
        return {
            'image': image,
            'text_feat': text_feat,
            'label': label
        }


# ====================== RUN ======================
train_fusion = FusionDataset(DATA_DIR, transform=transform)

🔍 Scanning cache for 3482 images...
📊 Cache Status: 3482 / 3482 images already processed (100.0%)
⏭️ Remaining to process: 0 images


📋 Empty / Very Short Text Files: 86
Some examples: ['00064657.txt', '2023182617.txt', '2028723868.txt', '2029242669.txt', '2030113468.txt']
🚀 Loading all cached features into memory (this may take 1-3 minutes)...



Loading Features:  39%|███▉      | 1358/3482 [27:42<36:22,  1.03s/it]  

In [ ]:
val_fusion = FusionDataset(DATA_DIR, transform=transform)

val_fusion.image_dataset = torch.utils.data.Subset(
    val_fusion.image_dataset, 
    val_dataset.indices
)

print(f"Train samples: {len(train_fusion)}")
print(f"Val samples:   {len(val_fusion)}")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_fusion, 
    batch_size=32, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True
)

val_loader = DataLoader(
    val_fusion, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

print("DataLoaders created successfully!")
print(f"Number of train batches: {len(train_loader)}")
print(f"Number of val batches:   {len(val_loader)}")

In [ ]:
import torch.nn as nn

class FusionClassifier(nn.Module):
    def __init__(self, num_classes=10, img_feat_dim=512, text_feat_dim=768, dropout=0.3):
        super().__init__()
        
        # Project image features (from ResNet)
        self.img_proj = nn.Sequential(
            nn.Linear(img_feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Project text features (from BERT)
        self.text_proj = nn.Sequential(
            nn.Linear(text_feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, image_feat, text_feat):
        img = self.img_proj(image_feat)      # [batch, 512]
        txt = self.text_proj(text_feat)      # [batch, 512]
        fused = torch.cat([img, txt], dim=1) # [batch, 1024]
        return self.classifier(fused)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import os

class FusionTrainer:
    def __init__(self, model, train_loader, val_loader, device=None,
                 num_epochs=15, lr=1e-4, weight_decay=1e-5,
                 patience=5, save_path=None):
        
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_epochs = num_epochs
        self.save_path = save_path or "/content/drive/MyDrive/document_classifier/best_fusion_model.pth"
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode='min', factor=0.5, 
                                           patience=3, min_lr=1e-6, verbose=True)
        
        self.start_epoch = 0
        self.best_val_acc = 0.0
        self.early_stop_patience = patience
        self.early_stop_counter = 0
        
        # Try to resume from checkpoint
        self.load_checkpoint()

    def load_checkpoint(self):
        if os.path.exists(self.save_path):
            print(f"🔄 Loading checkpoint from {self.save_path}")
            checkpoint = torch.load(self.save_path, map_location=self.device)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            self.start_epoch = checkpoint.get('epoch', 0) + 1
            self.best_val_acc = checkpoint.get('best_val_acc', 0.0)
            print(f"✅ Resumed from epoch {self.start_epoch} | Best Val Acc: {self.best_val_acc:.2f}%")
        else:
            print("🚀 Starting fresh training...")

    def save_checkpoint(self, epoch, val_acc):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_acc': val_acc
        }
        torch.save(checkpoint, self.save_path)
        print(f"💾 Best model saved at epoch {epoch} (Val Acc: {val_acc:.2f}%)")

    def train_one_epoch(self):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(self.train_loader, desc="Training", leave=False)
        for batch in loop:
            images = batch['image'].to(self.device)
            text_feats = batch['text_feat'].to(self.device)
            labels = batch['label'].to(self.device)

            # Get frozen ResNet features
            with torch.no_grad():
                img_feats = model_resnet(images)   # ← your feature extractor

            self.optimizer.zero_grad()
            outputs = self.model(img_feats, text_feats)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            loop.set_postfix(loss=loss.item())

        return running_loss / total, correct / total * 100

    def validate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validating", leave=False):
                images = batch['image'].to(self.device)
                text_feats = batch['text_feat'].to(self.device)
                labels = batch['label'].to(self.device)

                img_feats = model_resnet(images)
                outputs = self.model(img_feats, text_feats)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * labels.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        return running_loss / total, correct / total * 100

    def train(self):
        for epoch in range(self.start_epoch, self.start_epoch + self.num_epochs):
            print(f"\n{'='*60}\nEpoch {epoch+1}/{self.start_epoch + self.num_epochs}\n{'='*60}")
            
            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate()

            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Val Loss:   {val_loss:.4f}   | Val Acc:   {val_acc:.2f}%")

            # Scheduler step
            self.scheduler.step(val_loss)

            # Save best model
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.early_stop_counter = 0
                self.save_checkpoint(epoch, val_acc)
            else:
                self.early_stop_counter += 1
                print(f"Early stop counter: {self.early_stop_counter}/{self.early_stop_patience}")

            if self.early_stop_counter >= self.early_stop_patience:
                print("🛑 Early stopping triggered!")
                break

        print(f"\n🎉 Training finished! Best Validation Accuracy: {self.best_val_acc:.2f}%")

In [ ]:
# Make sure your ResNet feature extractor is ready (from earlier cells)
model_resnet = model  # the one where you did model.fc = nn.Identity()
model_resnet.eval()
model_resnet = model_resnet.to(device)

# Create fusion model
fusion_model = FusionClassifier(num_classes=len(train_fusion.classes))

# Create trainer
trainer = FusionTrainer(
    model=fusion_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,           # you can change
    lr=1e-4,
    patience=6,
    save_path="/content/drive/MyDrive/document_classifier/best_fusion_model.pth"
)

# Start training
trainer.train()